# Case 06 — **RobustScaler** vs Standard vs MinMax  ·  *Bai L5 (p7)*

## Y tuong
Ba cach dua feature ve cung thang do:

| Scaler | Cong thuc | Nhay outlier? |
|---|---|---|
| **Standard** | (x − trung binh) / do lech chuan | co (trung binh/do lech bi outlier keo) |
| **Robust** | (x − trung vi) / IQR | it (dung trung vi) |
| **MinMax** | ep ve [0, 1] | rat nhay |

## Vi sao chay thu nghiem nay
Du lieu co **diem cai cam clipping** (gia tri bi cat don o bien -> outlier nhan tao). Ve nguyen tac, khi co
outlier thi Robust an toan hon. Ta kiem bang so. *(Xet tren LogReg — cay/rung bat bien voi scale nen khong can.)*

In [1]:
# ============================================================================
# PRELUDE DUNG CHUNG cho moi notebook trong thu muc nay.
# Muc dich: moi file TU CHAY DUOC ma khong phu thuoc notebook chinh.
#   - Nap train.csv (Day chuyen A) + test.csv (Day chuyen B)
#   - Tao lai dung 4 feature co che nhu notebook chinh (de ket qua nhat quan)
# ============================================================================
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)   # co dinh seed -> chay lai ra so giong nhau

# --- Nap du lieu: thu nhieu duong dan vi notebook nam trong thu muc con ---
CANDIDATES = ['../Data_Final/Data_Final','../Data_Final','Data_Final/Data_Final',
              '../../Data_Final/Data_Final','Data_Final','.']
DATA_DIR = next((c for c in CANDIDATES
                 if os.path.exists(os.path.join(c,'train.csv'))
                 and os.path.exists(os.path.join(c,'test.csv'))), None)
assert DATA_DIR is not None, 'Khong tim thay train.csv/test.csv'
train = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))

NUM = ['nhiet_do_moi_truong','nhiet_do_quy_trinh','toc_do_quay','momen_xoan','do_mon_dao']  # 5 bien so goc
CAT = ['loai_san_pham','ca_lam_viec']   # 2 bien phan loai goc
TARGET = 'hong_hoc'                      # nhan: 1 = hong trong ca ke tiep

# --- Feature Engineering theo CO CHE vat ly (giong het notebook chinh) ---
def add_features(df):
    d = df.copy()
    # (1) Chenh lech nhiet: co che tan nhiet kem HDF. Lay HIEU nen triet tieu offset khi hau -> giam shift.
    d['chenh_lech_nhiet'] = d['nhiet_do_quy_trinh'] - d['nhiet_do_moi_truong']
    # (2) Cong suat co (W) = momen(Nm) * van_toc_goc(rad/s); van_toc_goc = rpm*2pi/60. Co che qua tai cong suat PWF.
    d['cong_suat_co']     = d['momen_xoan'] * d['toc_do_quay'] * 2*np.pi/60.0
    # (3) Tich mon*momen: co che qua tai cang thang OSF (mon cang nhieu + momen cang lon -> cang de gay).
    d['tich_mon_momen']   = d['do_mon_dao'] * d['momen_xoan']
    # (4) osf_margin: khoang cach toi NGUONG OSF, nguong phu thuoc HANG SP (L/M/H). >0 = da vuot nguong.
    g = d['loai_san_pham'].map({'L':11000,'M':12000,'H':13000})
    d['osf_margin']       = d['tich_mon_momen'] - g
    return d

train_fe = add_features(train); test_fe = add_features(test)
y_train = train_fe[TARGET].values
y_test  = test_fe[TARGET].values

# Bo feature CUOI da chot o notebook chinh: 9 bien so + 1 bien phan loai (loai_san_pham)
FINAL_NUM = NUM + ['chenh_lech_nhiet','cong_suat_co','tich_mon_momen','osf_margin']
FINAL_CAT = ['loai_san_pham']
print('Train:', train.shape, '| Test:', test.shape,
      '| ti le hong train/test:', round(y_train.mean(),3), round(y_test.mean(),3))
print('FINAL_NUM =', FINAL_NUM)

Train: (14000, 8) | Test: (6000, 8) | ti le hong train/test: 0.074 0.08
FINAL_NUM = ['nhiet_do_moi_truong', 'nhiet_do_quy_trinh', 'toc_do_quay', 'momen_xoan', 'do_mon_dao', 'chenh_lech_nhiet', 'cong_suat_co', 'tich_mon_momen', 'osf_margin']


In [2]:
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, average_precision_score
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

def run(scaler, name):
    # Chi thay doi SCALER, giu nguyen moi thu khac -> so sanh cong bang
    pre  = ColumnTransformer([('num', scaler, FINAL_NUM),
           ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), FINAL_CAT)])
    lr   = LogisticRegression(max_iter=5000, class_weight='balanced')
    pipe = Pipeline([('pre', pre), ('lr', lr)])
    oof  = cross_val_predict(pipe, train_fe[FINAL_NUM+FINAL_CAT], y_train, cv=skf, method='predict_proba', n_jobs=-1)[:,1]
    ts   = np.linspace(0.05,0.9,80); thr = ts[int(np.argmax([f1_score(y_train,(oof>=t)) for t in ts]))]
    pipe.fit(train_fe[FINAL_NUM+FINAL_CAT], y_train); p = pipe.predict_proba(test_fe[FINAL_NUM+FINAL_CAT])[:,1]
    return {'scaler':name, 'AUC_PR_test':average_precision_score(y_test,p),
            'F1_test':f1_score(y_test,(p>=thr)), 'AUC_PR_cv':average_precision_score(y_train,oof)}

rows = [run(StandardScaler(), 'Standard (mean/std)'),
        run(RobustScaler(),   'Robust (median/IQR)'),
        run(MinMaxScaler(),   'MinMax (min-max)')]
display(pd.DataFrame(rows).set_index('scaler').round(4))

,AUC_PR_test,F1_test,AUC_PR_cv
scaler,,,
Standard (mean/std),0.2444,0.3060,0.2625
Robust (median/IQR),0.2438,0.3083,0.2627
MinMax (min-max),0.2326,0.3007,0.2654


In [3]:
# Vi sao (khong) can Robust: dem 'outlier' (pile-up bien do clipping) tren tung feature THO
from scipy import stats
print('Ty le diem |z|>4 (outlier manh) va max|z| tren tung feature TRAIN:')
for c in NUM:
    z = np.abs(stats.zscore(train_fe[c]))
    print(f'  {c:20s} {(z>4).mean()*100:5.2f}%   (max|z|={z.max():.1f})')
print('\nGhi chu: bi CLIP o bien nen max|z| bi chan (~4), khong co duoi cuc doan -> chenh lech scaler nho.')

Ty le diem |z|>4 (outlier manh) va max|z| tren tung feature TRAIN:
  nhiet_do_moi_truong   0.01%   (max|z|=4.0)
  nhiet_do_quy_trinh    0.01%   (max|z|=4.0)
  toc_do_quay           0.00%   (max|z|=3.5)
  momen_xoan            0.00%   (max|z|=3.7)
  do_mon_dao            0.00%   (max|z|=1.7)

Ghi chu: bi CLIP o bien nen max|z| bi chan (~4), khong co duoi cuc doan -> chenh lech scaler nho.


> ### 🔎 Doc ket qua (Case 06)
> | Scaler | F1_test | AUC-PR_test |
> |---|---|---|
> | Standard | 0.306 | 0.244 |
> | **Robust** | **0.308** | 0.244 |
> | MinMax | 0.301 | 0.233 |
>
> Chenh lech **rat nho**: Robust ≥ Standard ≥ MinMax. Dem outlier cung it (bi clip o |z|≈4, khong co duoi cuc doan)
> nen loi the cua Robust khong lon.
>
> ### ✅ Ket luan Case 06
> Voi du lieu **bi clip**, **RobustScaler co ly le hon** ve nguyen tac (it bi keo boi bien). Nhung khac biet nho
> vi LogReg + class_weight von da ben; va voi **mo hinh cuoi (cay/rung) thi thang do KHONG quan trong chut nao**.
> Thong diep: chon scaler phai co **ly do gan voi dac diem du lieu**, khong chon theo quan tinh.